**MongoDB**

# MongoDB Assignment: Answers

Part A: 23 Theoretical Questions | Part B: 11 Practical Questions (PyMongo + Superstore dataset)

## How to run this notebook

Explanations are in text (Markdown) cells and all code is in code cells. The code is written in **Python with PyMongo**, so the mongosh examples from the theory answers appear here in their PyMongo form.

- You need a running MongoDB server (local at `localhost:27017`, or an Atlas cluster) and the setup cell below.
- Theory examples that query the `Orders` collection (Questions 6, 7, 11 and 21) need the data loaded first. Run **Practical Question 1** (load the CSV) before them, otherwise they run on an empty collection.
- The transaction example (Theory Question 10) needs a replica set, and the user example (Question 22) needs a server with authentication enabled.

In [ ]:
# Setup (run once)
%pip install --quiet pymongo pandas

from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
# For Atlas use your own connection string, for example:
# client = MongoClient("mongodb+srv://<user>:<password>@<cluster>.mongodb.net/")

db = client["superstore_db"]
orders = db["Orders"]

print("Connected. MongoDB version:", client.server_info()["version"])

---
# Part A: Theoretical Questions

## Question 1

> What are the key differences between SQL and NoSQL databases?

SQL databases are relational: data is stored in tables with fixed schemas. NoSQL databases store data in non-tabular models (documents, key-value pairs, wide columns or graphs) with flexible schemas.

| Feature | SQL (relational) | NoSQL (e.g. MongoDB) |
|---|---|---|
| Data model | Tables with rows and columns | Documents, key-value, column-family or graph |
| Schema | Fixed, predefined schema | Dynamic / flexible schema |
| Scalability | Mainly vertical (bigger server); horizontal scaling is harder | Designed for horizontal scaling (sharding) |
| Query language | Standard SQL | Database-specific APIs (MongoDB Query Language) |
| Relationships | Joins across normalized tables | Embedding or references; limited joins (`$lookup`) |
| Transactions | Strong ACID support as a core feature | Varies; MongoDB supports multi-document ACID transactions since 4.0 |
| Best suited for | Structured data, complex joins, banking and ERP systems | Big data, rapidly changing data, real-time and content-driven apps |
| Examples | MySQL, PostgreSQL, Oracle, SQL Server | MongoDB, Cassandra, Redis, Neo4j |

## Question 2

> What makes MongoDB a good choice for modern applications?

- **Flexible document model:** data is stored as JSON-like (BSON) documents that map naturally to objects in application code, and the schema can evolve without migrations.
- **Horizontal scalability:** built-in sharding spreads data across many servers.
- **High availability:** replica sets provide automatic failover and data redundancy.
- **Rich querying:** powerful query language, aggregation framework, secondary indexes, text and geospatial search.
- **High performance:** embedded documents reduce the need for joins, and indexing speeds up reads.
- **Multi-document ACID transactions** when strong consistency is required.
- **Cloud-ready:** MongoDB Atlas offers a managed service on AWS, Azure and Google Cloud.
- **Wide language support:** official drivers for Python, Java, Node.js, C#, Go and more.

## Question 3

> Explain the concept of collections in MongoDB.

A **collection** is a group of MongoDB documents. It is the equivalent of a table in a relational database, and it lives inside a database. Unlike a table, a collection does not enforce a fixed structure: documents in the same collection can have different fields, although in practice they usually share a similar shape.

- A collection is created implicitly the first time a document is inserted, or explicitly with `create_collection()` (`db.createCollection()` in the Mongo shell).
- Each document has a unique `_id` field, and every collection has an automatic index on `_id`.
- Collections can have indexes, validation rules and special options (for example capped collections).

In [ ]:
# explicit creation (only if it does not exist yet)
if "Orders" not in db.list_collection_names():
    db.create_collection("Orders")

# implicit creation: a collection is created on the first insert
db["Sales"].insert_one({"item": "Chair", "amount": 261.96})

print(db.list_collection_names())

## Question 4

> How does MongoDB ensure high availability using replication?

MongoDB uses **replica sets**: a group of `mongod` instances that all hold the same data. One member is the **primary**, which receives all writes, and the others are **secondaries** that copy the primary's changes.

- **Oplog:** the primary records every change in its operation log (oplog). Secondaries continuously read and apply these operations, so they stay up to date (replication is asynchronous by default).
- **Automatic failover:** if the primary becomes unreachable, the remaining members hold an election and promote one secondary to be the new primary, usually within seconds, without manual intervention.
- **Heartbeats:** members send heartbeats to each other to detect failures.
- **Data redundancy:** several copies of the data exist on different servers, so a single failure does not cause data loss.
- **Write concern and read preference:** `w: "majority"` makes a write durable across most members, and secondaries can serve reads if allowed.

A replica set should have at least three voting members (for example one primary and two secondaries) so a majority can always be formed to elect a primary.

## Question 5

> What are the main benefits of MongoDB Atlas?

MongoDB Atlas is MongoDB's fully managed cloud database service (Database-as-a-Service).

- **Fully managed:** automated provisioning, patching, upgrades and configuration; no server administration.
- **Multi-cloud:** runs on AWS, Microsoft Azure and Google Cloud, including global clusters across regions.
- **Easy scaling:** scale up or down, enable auto-scaling and add shards with a few clicks.
- **Built-in high availability:** every cluster is a replica set spread across availability zones.
- **Backups and recovery:** continuous cloud backups with point-in-time restore.
- **Security:** authentication, IP access lists, private networking (VPC peering / private endpoints), encryption in transit and at rest.
- **Monitoring and tuning:** real-time metrics, alerts and the Performance Advisor for index suggestions.
- **Extra services:** Atlas Search, Vector Search, Charts, Data Federation, and a free tier (M0) for learning.

## Question 6

> What is the role of indexes in MongoDB, and how do they improve performance?

An **index** is a special data structure (a B-tree) that stores the values of one or more fields in a sorted, easy-to-search form, with pointers to the actual documents. Without an index MongoDB must scan every document in the collection (a **collection scan**, `COLLSCAN`) to answer a query. With a suitable index it jumps directly to the matching entries (an **index scan**, `IXSCAN`), which greatly reduces the number of documents examined.

**Common index types**

- **Single field** and **compound** indexes (several fields).
- **Multikey** (on array fields), **text**, **geospatial** (2dsphere), **hashed**.
- **Unique**, **partial**, **sparse** and **TTL** (auto-expiring) indexes.

In [ ]:
orders.create_index([("Region", 1)])                     # single field
orders.create_index([("Region", 1), ("Sales", -1)])      # compound

print(orders.index_information())

**Trade-off:** indexes speed up reads and sorting, but they use extra storage and add a small cost to every insert, update and delete, so only fields used in queries should be indexed.

## Question 7

> Describe the stages of the MongoDB aggregation pipeline.

The **aggregation pipeline** processes documents through an ordered sequence of stages. The output of one stage becomes the input of the next, so data is filtered, reshaped, grouped and sorted step by step. It is called with `aggregate([ stage1, stage2, ... ])`.

| Stage | Purpose |
|---|---|
| `$match` | Filters documents (like WHERE); should be placed early so it can use indexes |
| `$group` | Groups documents by a key and computes aggregates such as `$sum`, `$avg`, `$min`, `$max`, `$count` |
| `$project` | Includes, excludes or reshapes fields, and can create computed fields |
| `$sort` | Sorts documents in ascending (1) or descending (-1) order |
| `$limit` / `$skip` | Restricts the number of documents / skips a number of documents |
| `$unwind` | Deconstructs an array field into one document per array element |
| `$lookup` | Performs a left outer join with another collection |
| `$addFields` / `$set` | Adds new fields to documents |
| `$count` | Returns the number of documents at that stage |
| `$out` / `$merge` | Writes the pipeline result to a collection |

**Example: total sales per category in the West region**

In [ ]:
pipeline = [
    {"$match": {"Region": "West"}},
    {"$group": {"_id": "$Category", "totalSales": {"$sum": "$Sales"}}},
    {"$sort": {"totalSales": -1}},
]

for row in orders.aggregate(pipeline):
    print(row)

## Question 8

> What is sharding in MongoDB? How does it differ from replication?

**Sharding** is MongoDB's method of horizontal scaling. A large collection is split into pieces (chunks) based on a **shard key**, and the pieces are distributed across multiple servers called **shards**. A sharded cluster has three components: **shards** (store the data, each usually a replica set), **mongos** routers (direct queries to the correct shard) and **config servers** (store cluster metadata). A balancer keeps chunks evenly distributed.

| Aspect | Sharding | Replication |
|---|---|---|
| Purpose | Scale storage and throughput | High availability and redundancy |
| Data on each node | A different subset of the data | A full copy of the same data |
| Scaling type | Horizontal scaling (write and storage capacity) | Mainly improves read availability; does not increase write capacity |
| Failure handling | Each shard should itself be a replica set | Automatic failover to a secondary |
| Key concept | Shard key, chunks, mongos | Primary, secondaries, oplog |

In production the two are used together: each shard is a replica set.

## Question 9

> What is PyMongo, and why is it used?

**PyMongo** is the official Python driver for MongoDB. It lets Python programs connect to a MongoDB server (local or Atlas) and perform CRUD operations, run aggregations, create indexes, use transactions and manage databases and collections, using ordinary Python dictionaries for documents. It is installed with `pip install pymongo`.

In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
shop = client["shop"]

shop.customers.insert_one({"name": "Asha", "city": "Pune"})
print(shop.customers.find_one({"name": "Asha"}))

It is used to build Python applications (web apps, data pipelines, analytics scripts) that need to store and retrieve data in MongoDB. For asynchronous applications the companion driver is Motor.

## Question 10

> What are the ACID properties in the context of MongoDB transactions?

| Property | Meaning | In MongoDB |
|---|---|---|
| Atomicity | A transaction is all-or-nothing | Operations on a single document are always atomic; multi-document transactions commit or roll back as a whole |
| Consistency | Data moves from one valid state to another | Maintained through schema validation, unique indexes and the rules applied when a transaction commits |
| Isolation | Concurrent transactions do not see each other's uncommitted changes | Transactions use snapshot isolation |
| Durability | Committed data survives failures | Ensured by journaling and write concern (for example `w: "majority"`) |

Multi-document ACID transactions are available from MongoDB 4.0 on replica sets and from 4.2 on sharded clusters. Example with PyMongo (needs a replica set):

In [ ]:
accounts = client["bank"]["accounts"]

with client.start_session() as session:
    with session.start_transaction():
        accounts.update_one({"_id": "A"},
                            {"$inc": {"balance": -100}}, session=session)
        accounts.update_one({"_id": "B"},
                            {"$inc": {"balance": 100}}, session=session)

## Question 11

> What is the purpose of MongoDB's explain() function?

`explain()` returns details about how MongoDB executes a query, called the **query plan**. It is the main tool for diagnosing slow queries and checking whether an index is being used.

- **queryPlanner** (default): shows the winning plan without running the query.
- **executionStats**: runs the query and reports actual statistics.
- **allPlansExecution**: also reports the rejected candidate plans.

In [ ]:
plan = db.command({
    "explain": {"find": "Orders", "filter": {"Region": "West"}},
    "verbosity": "executionStats",
})

print("Winning plan:", plan["queryPlanner"]["winningPlan"])
stats = plan["executionStats"]
print("nReturned:", stats["nReturned"])
print("totalKeysExamined:", stats["totalKeysExamined"])
print("totalDocsExamined:", stats["totalDocsExamined"])
print("executionTimeMillis:", stats["executionTimeMillis"])

Important fields: the stage in the winning plan (`COLLSCAN` = full scan, `IXSCAN` = index used), `nReturned`, `totalKeysExamined`, `totalDocsExamined` and `executionTimeMillis`. A large gap between documents examined and documents returned suggests a missing or poor index.

## Question 12

> How does MongoDB handle schema validation?

MongoDB is schema-flexible, but it lets you enforce rules on a collection through a **validator**, usually written with `$jsonSchema`. It can be set when the collection is created or added later with `collMod`.

- **validationLevel:** `strict` (default, validates all inserts and updates), `moderate` (validates only documents that already conform) or `off`.
- **validationAction:** `error` (reject the invalid write) or `warn` (allow it but log a warning).

In [ ]:
db.drop_collection("customers")

db.create_collection(
    "customers",
    validator={"$jsonSchema": {
        "bsonType": "object",
        "required": ["name", "email"],
        "properties": {
            "name":  {"bsonType": "string"},
            "email": {"bsonType": "string", "pattern": "^.+@.+$"},
            "age":   {"bsonType": "int", "minimum": 18},
        },
    }},
    validationLevel="strict",
    validationAction="error",
)

Test the validator: a valid document is accepted and an invalid one is rejected.

In [ ]:
from pymongo.errors import WriteError

db.customers.insert_one({"name": "Asha", "email": "asha@example.com", "age": 25})
print("Valid document inserted")

try:
    db.customers.insert_one({"name": "Ravi"})        # email is missing
except WriteError as e:
    print("Rejected by validation:", e.details.get("errmsg"))

## Question 13

> What is the difference between a primary and a secondary node in a replica set?

| Aspect | Primary | Secondary |
|---|---|---|
| Role | The only node that accepts write operations | Maintains a copy of the primary's data |
| Number | Exactly one at a time | One or more |
| Data source | Records changes in the oplog | Replicates the primary's oplog and applies it |
| Reads | Serves reads by default | Serves reads only if the read preference allows it |
| Failure | If it fails, an election is held | Can be elected as the new primary |

Secondaries can also be configured as priority-0, hidden or delayed members for special purposes such as backups or reporting.

## Question 14

> What security mechanisms does MongoDB provide for data protection?

- **Authentication:** verifies identity using SCRAM (default), x.509 certificates, and LDAP or Kerberos in MongoDB Enterprise.
- **Authorization (RBAC):** role-based access control with built-in and custom roles, so users get only the privileges they need.
- **Encryption in transit:** TLS/SSL for client connections and between cluster members.
- **Encryption at rest:** encrypted storage engine (Enterprise) and default in Atlas.
- **Field-level protection:** Client-Side Field Level Encryption and Queryable Encryption keep sensitive fields encrypted even from the server.
- **Auditing:** logs access and administrative actions (Enterprise / Atlas).
- **Network security:** bind IP restrictions, firewalls, IP access lists, VPC peering and private endpoints.

## Question 15

> Explain the concept of embedded documents and when they should be used.

An **embedded document** is a document stored inside another document as a field value or as an array of documents. It lets related data be kept together in one record instead of split across collections.

In [ ]:
order = {
    "_id": 1,
    "customer": "Asha",
    "address": {"city": "Pune", "pin": "411001"},   # embedded document
    "items": [                                      # array of documents
        {"product": "Laptop", "qty": 1},
        {"product": "Mouse", "qty": 2},
    ],
}

db["embedded_demo"].replace_one({"_id": 1}, order, upsert=True)

saved = db["embedded_demo"].find_one({"_id": 1})
print(saved["address"]["city"], "|", len(saved["items"]), "items")

**Use embedding when**

- The data has a one-to-one or one-to-few relationship (address, order items).
- The related data is usually read together, so one query returns everything with no join.
- The child data belongs to the parent and is updated together with it (single-document updates are atomic).

**Avoid embedding when**

- The embedded array can grow without limit (documents are limited to 16 MB).
- The child data is shared by many parents or changes independently (use references instead).

## Question 16

> What is the purpose of MongoDB's `$lookup` stage in aggregation?

`$lookup` performs a **left outer join** between the collection being aggregated and another collection in the same database. For each input document it adds an array field containing the matching documents from the other collection. It is used when data is stored in separate collections (referenced rather than embedded).

In [ ]:
pipeline = [
    {"$lookup": {
        "from": "customers",            # collection to join
        "localField": "customerId",     # field in Orders
        "foreignField": "_id",          # field in customers
        "as": "customerDetails",        # output array field
    }},
    {"$unwind": "$customerDetails"},    # optional: array to object
]

result = list(orders.aggregate(pipeline))
print(len(result), "joined documents")

The collection and field names above are illustrative: the Superstore data has no separate customers collection with a `customerId` field, so this example returns no joined documents. It shows the syntax to use when two related collections exist.

Documents with no match get an empty array. Because joins are more expensive than embedded reads, `$lookup` should be used selectively.

## Question 17

> What are some common use cases for MongoDB?

- **E-commerce and product catalogs:** products with different attributes fit flexible documents.
- **Content management and blogging platforms.**
- **Real-time analytics and dashboards.**
- **Internet of Things (IoT) and time-series data:** high-volume sensor data.
- **Mobile and web applications:** user profiles, sessions and preferences.
- **Personalization and recommendation engines.**
- **Gaming:** player profiles, leaderboards and game state.
- **Social networks and messaging:** feeds, comments, chats.
- **Logging and event data**, and a "single view" of customer data collected from many systems.

## Question 18

> What are the advantages of using MongoDB for horizontal scaling?

- **Built-in sharding:** data is distributed across servers automatically using a shard key.
- **Higher capacity and throughput:** storage, reads and writes grow by adding more machines instead of upgrading one large server.
- **Cost-effective:** uses many commodity servers, which is cheaper than very large single machines.
- **Transparent to applications:** `mongos` routers direct queries, so application code changes little.
- **Automatic balancing:** the balancer moves chunks between shards to keep data even.
- **Zone sharding:** data can be placed in specific regions to reduce latency or meet data-residency rules.
- **Combined with replication:** every shard is a replica set, so scaling does not cost availability.

## Question 19

> How do MongoDB transactions differ from SQL transactions?

| Aspect | MongoDB | SQL databases |
|---|---|---|
| Basic unit | Operations on a single document are atomic by default | Transactions are central and span rows and tables |
| Multi-record transactions | Available since 4.0 (replica sets) and 4.2 (sharded clusters); needed less often because of embedding | Always supported and commonly used |
| Syntax | Sessions: `start_session()`, `start_transaction()`, `commit_transaction()` | `BEGIN`, `COMMIT`, `ROLLBACK` |
| Isolation | Snapshot isolation | Configurable levels (Read Committed, Repeatable Read, Serializable, ...) |
| Conflicts | A write conflict aborts the transaction, which the application can retry | Usually handled with locks or waiting |
| Limits and cost | Adds overhead; default 60 second lifetime limit | Highly optimized for transactional workloads |

## Question 20

> What are the main differences between capped collections and regular collections?

A **capped collection** is a fixed-size collection that keeps documents in insertion order and automatically overwrites the oldest documents when it is full (like a circular buffer).

In [ ]:
db.drop_collection("appLogs")
db.create_collection("appLogs", capped=True, size=1048576, max=1000)

print(db["appLogs"].options())

| Feature | Capped collection | Regular collection |
|---|---|---|
| Size | Fixed maximum size (and optionally document count), set at creation | Grows as needed |
| When full | Oldest documents are removed automatically | No automatic removal |
| Order | Insertion order is preserved | No guaranteed natural order |
| Deleting documents | Individual documents cannot normally be deleted | Documents can be deleted freely |
| Sharding | Not supported | Supported |
| Typical use | Logs, cache, recent-activity feeds | General application data |

## Question 21

> What is the purpose of the `$match` stage in MongoDB's aggregation pipeline?

`$match` **filters** the documents flowing through the pipeline, passing only those that satisfy the given condition to the next stage. It uses the same query syntax as `find()` and works like the WHERE clause in SQL.

- Placing `$match` **early** in the pipeline reduces the number of documents later stages must process.
- When it is the first stage, it can use indexes for faster execution.

In [ ]:
pipeline = [
    {"$match": {"Region": "West", "Sales": {"$gt": 500}}},
    {"$group": {"_id": "$Category", "total": {"$sum": "$Sales"}}},
]

for row in orders.aggregate(pipeline):
    print(row)

## Question 22

> How can you secure access to a MongoDB database?

- **Enable authentication** (`security.authorization: enabled` in the config file, or start `mongod --auth`).
- **Use role-based access control** and follow least privilege: give each user only the roles they need.
- **Enable TLS/SSL** to encrypt data in transit.
- **Restrict network access:** bind to specific IPs, use firewalls, never expose port 27017 to the public internet; in Atlas use IP access lists and private endpoints.
- **Encrypt data at rest** and consider field-level encryption for sensitive data.
- **Enable auditing**, use strong passwords and keep MongoDB updated with security patches.

Example: create a user who can only read and write one database (run as an administrator; choose your own strong password):

In [ ]:
client.admin.command(
    "createUser", "appUser",
    pwd="Replace-With-A-Strong-Password",
    roles=[{"role": "readWrite", "db": "superstore_db"}],
)

## Question 23

> What is MongoDB's WiredTiger storage engine, and why is it important?

**WiredTiger** is MongoDB's default storage engine (since version 3.2). The storage engine controls how data is stored on disk and in memory.

- **Document-level concurrency control:** many clients can write to different documents in the same collection at the same time, improving throughput.
- **Compression:** collections and indexes are compressed (Snappy by default; zlib and zstd are available), saving disk space and I/O.
- **Checkpoints and journaling:** data is written to disk at regular checkpoints and changes are recorded in a journal, giving durability and fast crash recovery.
- **Internal cache:** frequently used data is kept in memory (by default about 50% of RAM minus 1 GB).
- **Encryption at rest** support (Enterprise) and support for multi-document transactions.

---
# Part B: Practical Questions

**Dataset:** Superstore dataset (CSV). Column names such as `Region`, `Sales`, `Profit`, `Ship Mode` and `Category` follow the standard Superstore file; if your CSV uses different names, change them in the queries.

**Assumptions:** MongoDB runs at `localhost:27017` (or use an Atlas connection string), the database is named `superstore_db`, the collection is `Orders`, and the CSV file is `Superstore.csv` (placed next to this notebook). The connection variables `client`, `db` and `orders` come from the setup cell at the top.

Note: the questions are answered in the given order. Question 7 changes data and Question 8 deletes documents, so the results of Questions 9 to 11 reflect those changes.

## Practical Question 1

> Write a Python script to load the Superstore dataset from a CSV file into MongoDB.

In [ ]:
import pandas as pd
from pymongo import MongoClient

CSV_PATH = "Superstore.csv"

client = MongoClient("mongodb://localhost:27017/")
db = client["superstore_db"]
orders = db["Orders"]

# Read the CSV (latin-1 avoids decoding errors in the usual file)
df = pd.read_csv(CSV_PATH, encoding="latin-1")

# Convert each row into a dictionary (one document per row)
records = df.to_dict(orient="records")

orders.drop()          # start clean so re-running does not duplicate data
result = orders.insert_many(records)

print("Inserted documents:", len(result.inserted_ids))

`pandas` reads the CSV, `to_dict("records")` turns every row into a document, and `insert_many()` inserts all of them into the `Orders` collection in one call.

## Practical Question 2

> Retrieve and print all documents from the Orders collection.

In [ ]:
for doc in orders.find():
    print(doc)

Mongo shell: `db.Orders.find()`. The full dataset has thousands of documents, so use `orders.find().limit(5)` if you only want a quick preview.

## Practical Question 3

> Count and display the total number of documents in the Orders collection.

In [ ]:
total = orders.count_documents({})
print("Total documents in Orders:", total)

Mongo shell: `db.Orders.countDocuments({})`. For the standard Sample Superstore file this is 9,994 documents.

## Practical Question 4

> Write a query to fetch all orders from the "West" region.

In [ ]:
west_orders = list(orders.find({"Region": "West"}))
print("Orders in West region:", len(west_orders))
for doc in west_orders:
    print(doc)

Mongo shell: `db.Orders.find({ Region: "West" })`

## Practical Question 5

> Write a query to find orders where Sales is greater than 500.

In [ ]:
high_sales = list(orders.find({"Sales": {"$gt": 500}}))
print("Orders with Sales > 500:", len(high_sales))
for doc in high_sales:
    print(doc)

Mongo shell: `db.Orders.find({ Sales: { $gt: 500 } })`. `$gt` means "greater than".

## Practical Question 6

> Fetch the top 3 orders with the highest Profit.

In [ ]:
top3 = orders.find().sort("Profit", -1).limit(3)
for doc in top3:
    print(doc)

Mongo shell: `db.Orders.find().sort({ Profit: -1 }).limit(3)`. Sorting by -1 gives descending order, and `limit(3)` keeps the first three.

## Practical Question 7

> Update all orders with Ship Mode as "First Class" to "Premium Class."

In [ ]:
result = orders.update_many(
    {"Ship Mode": "First Class"},
    {"$set": {"Ship Mode": "Premium Class"}}
)
print("Matched:", result.matched_count, "| Modified:", result.modified_count)

Mongo shell: `db.Orders.updateMany({ "Ship Mode": "First Class" }, { $set: { "Ship Mode": "Premium Class" } })`. Field names that contain a space must be written in quotes.

## Practical Question 8

> Delete all orders where Sales is less than 50.

In [ ]:
result = orders.delete_many({"Sales": {"$lt": 50}})
print("Deleted documents:", result.deleted_count)

Mongo shell: `db.Orders.deleteMany({ Sales: { $lt: 50 } })`. `$lt` means "less than". This permanently removes the matching documents.

## Practical Question 9

> Use aggregation to group orders by Region and calculate total sales per region.

In [ ]:
pipeline = [
    {"$group": {"_id": "$Region", "TotalSales": {"$sum": "$Sales"}}},
    {"$sort": {"TotalSales": -1}}
]
for row in orders.aggregate(pipeline):
    print(row["_id"], round(row["TotalSales"], 2))

`$group` collects the orders of each region (`_id: "$Region"`) and `$sum` adds up their Sales. `$sort` then lists the regions from highest to lowest total. Mongo shell: `db.Orders.aggregate([{ $group: { _id: "$Region", TotalSales: { $sum: "$Sales" } } }])`

## Practical Question 10

> Fetch all distinct values for Ship Mode from the collection.

In [ ]:
ship_modes = orders.distinct("Ship Mode")
print(ship_modes)

Mongo shell: `db.Orders.distinct("Ship Mode")`. After Question 7 the list contains "Premium Class" instead of "First Class" (the other values are typically "Second Class", "Standard Class" and "Same Day").

## Practical Question 11

> Count the number of orders for each category.

In [ ]:
pipeline = [
    {"$group": {"_id": "$Category", "OrderCount": {"$sum": 1}}},
    {"$sort": {"OrderCount": -1}}
]
for row in orders.aggregate(pipeline):
    print(row["_id"], row["OrderCount"])

Each document adds 1 to its category's counter (`$sum: 1`). Mongo shell: `db.Orders.aggregate([{ $group: { _id: "$Category", OrderCount: { $sum: 1 } } }])`. The typical categories are Furniture, Office Supplies and Technology.